# 08. Correlation-Aware Survivor Freeze / Pre-ML Alpha Library v4

Freeze final production-path survivor alphas from Notebook 07, with an added correlation-aware selection layer so the final core set is strong, stress-approved, and not redundant.

Core production path:
04A Dynamic Alpha Engine -> 04B Alpha WFV -> 07 Alpha Stress Testing -> 08 Survivor Freeze -> 09 Portfolio Construction.

Scope guardrails: this notebook does not modify alpha formulas, construction logic, WFV logic, stress logic, portfolio logic, or ML. It does not use 05/06 regime overlay outputs. Satellite and correlated alternate alphas may be retained in the registry for audit, but only final `PROMOTE_CORE` rows feed `pre_ml_alpha_inputs_current`.

## 1. Imports and Config

In [1]:
from pathlib import Path
import gc
import sys

import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif PROJECT_ROOT.name == "2-Phase 2_Signal Expansion":
    PROJECT_ROOT = PROJECT_ROOT.parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.db import load_table, table_exists
from src.run_config import get_sqlite_db_path, make_run_id, make_run_timestamp
from src.survivor_registry import (
    add_alpha_behavior_clusters,
    add_survivor_selection_scores,
    build_constructed_pre_ml_alpha_inputs,
    build_constructed_survivor_lineage_report,
    build_correlation_aware_survivor_registry,
    build_survivor_candidate_pool,
    build_survivor_cluster_summary,
    build_survivor_freeze_report,
    compute_survivor_alpha_correlations,
    validate_correlation_aware_survivor_registry,
)
from src.survivor_storage import SURVIVOR_TABLES, save_survivor_outputs

DB_PATH = get_sqlite_db_path()
SURVIVOR_VERSION = "phase8_cluster_aware_survivor_v5"
CORE_MAX_ABS_CORRELATION = 0.90
CLUSTER_MAX_ABS_CORRELATION = 0.95
TARGET_MAX_CORE_SURVIVORS = 4
MIN_CLUSTER_SCORE = 40
PRIMARY_PROMOTION_DECISIONS = ["PROMOTE_CORE", "PROMOTE_BALANCED", "REVIEW_SATELLITE"]

pd.set_option("display.max_columns", 200)
DB_PATH


PosixPath('/Users/AnyiXu_1/Desktop/multi-factor-equity-alpha-model/sql/project_underdog.db')

## 2. Create freeze run_id / timestamp

In [2]:
run_id = make_run_id(prefix="phase8_cluster_aware_survivor")
timestamp_frozen = make_run_timestamp()

run_id, timestamp_frozen


('phase8_cluster_aware_survivor_20260511_084437', '2026-05-11 08:44:37')

## 3. Load production-path inputs

In [3]:
alpha_stress_gate = load_table("alpha_stress_gate_current", db_path=DB_PATH)
alpha_stress_audit = load_table("alpha_stress_audit_summary_current", db_path=DB_PATH)
alpha_constructed_candidates = load_table("alpha_constructed_candidates_current", db_path=DB_PATH)
alpha_construction_metadata = load_table("alpha_construction_metadata_current", db_path=DB_PATH)
alpha_construction_diagnostics = load_table("alpha_construction_diagnostics_current", db_path=DB_PATH)
constructed_alpha_wfv_gate = load_table("constructed_alpha_wfv_gate_current", db_path=DB_PATH)
constructed_alpha_wfv_winner_summary = load_table("constructed_alpha_wfv_winner_summary_current", db_path=DB_PATH)
prior_survivor_registry = (
    load_table("survivor_alpha_registry_current", db_path=DB_PATH)
    if table_exists("survivor_alpha_registry_current", db_path=DB_PATH)
    else pd.DataFrame()
)

input_shapes = pd.DataFrame(
    [
        {"artifact": "alpha_stress_gate_current", "rows": len(alpha_stress_gate), "columns": len(alpha_stress_gate.columns)},
        {"artifact": "alpha_stress_audit_summary_current", "rows": len(alpha_stress_audit), "columns": len(alpha_stress_audit.columns)},
        {"artifact": "alpha_constructed_candidates_current", "rows": len(alpha_constructed_candidates), "columns": len(alpha_constructed_candidates.columns)},
        {"artifact": "alpha_construction_metadata_current", "rows": len(alpha_construction_metadata), "columns": len(alpha_construction_metadata.columns)},
        {"artifact": "alpha_construction_diagnostics_current", "rows": len(alpha_construction_diagnostics), "columns": len(alpha_construction_diagnostics.columns)},
        {"artifact": "constructed_alpha_wfv_gate_current", "rows": len(constructed_alpha_wfv_gate), "columns": len(constructed_alpha_wfv_gate.columns)},
        {"artifact": "constructed_alpha_wfv_winner_summary_current", "rows": len(constructed_alpha_wfv_winner_summary), "columns": len(constructed_alpha_wfv_winner_summary.columns)},
        {"artifact": "prior survivor_alpha_registry_current comparison only", "rows": len(prior_survivor_registry), "columns": len(prior_survivor_registry.columns)},
    ]
)
stress_candidate_counts = (
    alpha_stress_gate["promotion_decision"]
    .value_counts(dropna=False)
    .rename_axis("promotion_decision")
    .reset_index(name="n_alpha_horizons")
)

print("Stress candidate counts by promotion_decision")
display(stress_candidate_counts)
display(input_shapes)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after loading survivor freeze input tables')


Stress candidate counts by promotion_decision


,promotion_decision,n_alpha_horizons
0,REVIEW_SATELLITE,2
1,REJECT,1


,artifact,rows,columns
0,alpha_stress_gate_current,3,23
1,alpha_stress_audit_summary_current,3,24
2,alpha_constructed_candidates_current,10028440,6
3,alpha_construction_metadata_current,10,24
4,alpha_construction_diagnostics_current,10,16
5,constructed_alpha_wfv_gate_current,24,18
6,constructed_alpha_wfv_winner_summary_current,6,10
7,prior survivor_alpha_registry_current comparis...,3,29


## 4. Build scored survivor candidate pool

In [4]:
all_stress_labeled_candidates = build_survivor_candidate_pool(
    stress_gate=alpha_stress_gate,
    stress_audit=alpha_stress_audit,
    construction_metadata=alpha_construction_metadata,
    construction_diagnostics=alpha_construction_diagnostics,
    constructed_alpha_wfv_winner_summary=constructed_alpha_wfv_winner_summary,
)
tracked_candidate_mask = (
    all_stress_labeled_candidates["promotion_decision"].isin(PRIMARY_PROMOTION_DECISIONS)
    | all_stress_labeled_candidates["promotion_decision"].eq("REJECT_HIGH_TURNOVER")
    | all_stress_labeled_candidates["turnover_risk_flag"].astype(str).eq("HIGH_TURNOVER_RISK")
)
all_stress_labeled_candidates = all_stress_labeled_candidates.loc[tracked_candidate_mask].copy()
all_stress_labeled_candidates = add_alpha_behavior_clusters(
    add_survivor_selection_scores(all_stress_labeled_candidates)
)
primary_candidate_pool = all_stress_labeled_candidates.loc[
    all_stress_labeled_candidates["promotion_decision"].isin(PRIMARY_PROMOTION_DECISIONS)
].copy()

stress_gate_diagnostic_columns = [
    "alpha_name",
    "horizon",
    "status",
    "promotion_decision",
    "survivor_tier",
    "alpha_sleeve",
    "pass_rate",
    "worst_degradation",
    "turnover_risk_flag",
]
candidate_pool_diagnostic_columns = [
    "alpha_name",
    "horizon",
    "stress_status",
    "promotion_decision",
    "survivor_tier",
    "alpha_sleeve",
    "pass_rate",
    "worst_degradation",
    "turnover_risk_flag",
    "survivor_selection_score",
    "alpha_behavior_cluster",
    "cluster_rank",
    "source_wfv_status",
]
admission_check = pd.DataFrame(
    [
        {
            "promotion_decision": decision,
            "admitted_to_primary_candidate_pool": decision in PRIMARY_PROMOTION_DECISIONS,
            "n_rows_in_stress_gate": int(alpha_stress_gate["promotion_decision"].astype(str).eq(decision).sum()),
            "n_rows_admitted": int(primary_candidate_pool["promotion_decision"].astype(str).eq(decision).sum()),
        }
        for decision in ["PROMOTE_CORE", "PROMOTE_BALANCED", "REVIEW_SATELLITE", "REJECT_HIGH_TURNOVER", "REJECT"]
    ]
)

print("alpha_stress_gate_current candidate diagnostics")
display(
    alpha_stress_gate[[column for column in stress_gate_diagnostic_columns if column in alpha_stress_gate.columns]]
    .sort_values(["promotion_decision", "alpha_name", "horizon"])
    .reset_index(drop=True)
)
print("08 primary admission check")
display(admission_check)
print("Admitted primary_candidate_pool diagnostics")
display(
    primary_candidate_pool[[column for column in candidate_pool_diagnostic_columns if column in primary_candidate_pool.columns]]
    .sort_values(["promotion_decision", "alpha_name", "horizon"])
    .reset_index(drop=True)
)
scored_primary_candidates = primary_candidate_pool.sort_values(
    ["survivor_selection_score", "alpha_name"],
    ascending=[False, True],
).reset_index(drop=True)

behavior_cluster_counts = (
    primary_candidate_pool["alpha_behavior_cluster"]
    .value_counts(dropna=False)
    .rename_axis("alpha_behavior_cluster")
    .reset_index(name="n_alpha_horizons")
)
score_columns = candidate_pool_diagnostic_columns

print(f"Primary stress candidate pool rows: {len(primary_candidate_pool)}")
print("Behavior cluster counts")
display(behavior_cluster_counts)
display(scored_primary_candidates[[column for column in score_columns if column in scored_primary_candidates.columns]])


alpha_stress_gate_current candidate diagnostics


,alpha_name,horizon,status,promotion_decision,survivor_tier,pass_rate,worst_degradation,turnover_risk_flag
0,alpha_orthogonal_diversifier_v2_score_weighted...,20,REJECTED_STRESS,REJECT,NON_SURVIVOR,0.833333,0.897289,MODERATE_TURNOVER_RISK
1,alpha_hybrid_adaptive_v4_smooth,20,APPROVED_STRESS,REVIEW_SATELLITE,WATCH_STRESS_SURVIVOR,0.833333,0.678978,LOW_TURNOVER_RISK
2,alpha_regime_blend_dynamic_v4_smooth,20,APPROVED_STRESS,REVIEW_SATELLITE,WATCH_STRESS_SURVIVOR,0.888889,0.422319,LOW_TURNOVER_RISK


08 primary admission check


,promotion_decision,admitted_to_primary_candidate_pool,n_rows_in_stress_gate,n_rows_admitted
0,PROMOTE_CORE,True,0,0
1,PROMOTE_BALANCED,True,0,0
2,REVIEW_SATELLITE,True,2,2
3,REJECT_HIGH_TURNOVER,False,0,0
4,REJECT,False,1,0


Admitted primary_candidate_pool diagnostics


,alpha_name,horizon,stress_status,promotion_decision,survivor_tier,alpha_sleeve,pass_rate,worst_degradation,turnover_risk_flag,survivor_selection_score,alpha_behavior_cluster,cluster_rank,source_wfv_status
0,alpha_hybrid_adaptive_v4_smooth,20,APPROVED_STRESS,REVIEW_SATELLITE,WATCH_STRESS_SURVIVOR,CORE_REGIME,0.833333,0.678978,LOW_TURNOVER_RISK,37.667225,CORE_REGIME,2,WATCHLIST_CONSTRUCTED_ALPHA_WFV
1,alpha_regime_blend_dynamic_v4_smooth,20,APPROVED_STRESS,REVIEW_SATELLITE,WATCH_STRESS_SURVIVOR,CORE_REGIME,0.888889,0.422319,LOW_TURNOVER_RISK,49.596981,CORE_REGIME,1,WATCHLIST_CONSTRUCTED_ALPHA_WFV


Primary stress candidate pool rows: 2
Behavior cluster counts


,alpha_behavior_cluster,n_alpha_horizons
0,CORE_REGIME,2


,alpha_name,horizon,stress_status,promotion_decision,survivor_tier,alpha_sleeve,pass_rate,worst_degradation,turnover_risk_flag,survivor_selection_score,alpha_behavior_cluster,cluster_rank,source_wfv_status
0,alpha_regime_blend_dynamic_v4_smooth,20,APPROVED_STRESS,REVIEW_SATELLITE,WATCH_STRESS_SURVIVOR,CORE_REGIME,0.888889,0.422319,LOW_TURNOVER_RISK,49.596981,CORE_REGIME,1,WATCHLIST_CONSTRUCTED_ALPHA_WFV
1,alpha_hybrid_adaptive_v4_smooth,20,APPROVED_STRESS,REVIEW_SATELLITE,WATCH_STRESS_SURVIVOR,CORE_REGIME,0.833333,0.678978,LOW_TURNOVER_RISK,37.667225,CORE_REGIME,2,WATCHLIST_CONSTRUCTED_ALPHA_WFV


## 5. Compute alpha correlation diagnostics

In [5]:
survivor_alpha_correlation = compute_survivor_alpha_correlations(
    alpha_candidates_long=alpha_constructed_candidates,
    candidates=primary_candidate_pool,
    survivor_version=SURVIVOR_VERSION,
    run_id=run_id,
)

if survivor_alpha_correlation.empty:
    correlation_matrix = pd.DataFrame()
    highest_correlations = survivor_alpha_correlation
else:
    pairwise_for_matrix = survivor_alpha_correlation.copy()
    names = sorted(primary_candidate_pool["alpha_name"].dropna().unique())
    correlation_matrix = pd.DataFrame(1.0, index=names, columns=names)
    for row in pairwise_for_matrix.itertuples(index=False):
        correlation_matrix.loc[row.alpha_name_1, row.alpha_name_2] = row.correlation
        correlation_matrix.loc[row.alpha_name_2, row.alpha_name_1] = row.correlation
    highest_correlations = survivor_alpha_correlation.sort_values("abs_correlation", ascending=False).head(20)

print(f"Pairwise correlation rows: {len(survivor_alpha_correlation)}")
print("Correlation matrix")
display(correlation_matrix)
print("Highest absolute correlations")
display(highest_correlations)


Pairwise correlation rows: 1
Correlation matrix


,alpha_hybrid_adaptive_v4_smooth,alpha_regime_blend_dynamic_v4_smooth
alpha_hybrid_adaptive_v4_smooth,1.000000,0.990363
alpha_regime_blend_dynamic_v4_smooth,0.990363,1.000000


Highest absolute correlations


,alpha_name_1,alpha_name_2,correlation,abs_correlation,alpha_1_promotion_decision,alpha_2_promotion_decision,alpha_1_pass_rate,alpha_2_pass_rate,alpha_1_worst_degradation,alpha_2_worst_degradation,survivor_version,run_id
0,alpha_hybrid_adaptive_v4_smooth,alpha_regime_blend_dynamic_v4_smooth,0.990363,0.990363,REVIEW_SATELLITE,REVIEW_SATELLITE,0.833333,0.888889,0.678978,0.422319,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437


## 6. Select final core survivors and retained alternatives

In [6]:
survivor_registry = build_correlation_aware_survivor_registry(
    candidates=all_stress_labeled_candidates,
    alpha_correlations=survivor_alpha_correlation,
    survivor_version=SURVIVOR_VERSION,
    run_id=run_id,
    timestamp_frozen=timestamp_frozen,
    core_max_abs_correlation=CORE_MAX_ABS_CORRELATION,
    cluster_max_abs_correlation=CLUSTER_MAX_ABS_CORRELATION,
    target_max_core_survivors=TARGET_MAX_CORE_SURVIVORS,
    min_cluster_score=MIN_CLUSTER_SCORE,
)
survivor_cluster_summary = build_survivor_cluster_summary(
    survivor_registry=survivor_registry,
    alpha_correlations=survivor_alpha_correlation,
    survivor_version=SURVIVOR_VERSION,
    run_id=run_id,
)

final_core_registry = survivor_registry.loc[
    survivor_registry["promotion_decision_final"].eq("PROMOTE_CORE")
].copy()
retained_alternates = survivor_registry.loc[
    ~survivor_registry["promotion_decision_final"].eq("PROMOTE_CORE")
].copy()

final_status_counts = (
    survivor_registry["final_status"].value_counts(dropna=False).rename_axis("final_status").reset_index(name="n_rows")
)
promotion_final_counts = (
    survivor_registry["promotion_decision_final"].value_counts(dropna=False).rename_axis("promotion_decision_final").reset_index(name="n_rows")
)

print(f"Final PROMOTE_CORE survivors selected: {len(final_core_registry)}")
print("Cluster summary")
display(survivor_cluster_summary)
print("Final registry by final_status")
display(final_status_counts)
print("Final registry by promotion_decision_final")
display(promotion_final_counts)
print("Final core selected alphas")
display(final_core_registry)
print("Satellite/watchlist/alternate alphas")
display(retained_alternates)

admitted_final_decisions = survivor_registry.merge(
    primary_candidate_pool[["alpha_name", "horizon"]].drop_duplicates(),
    on=["alpha_name", "horizon"],
    how="inner",
)
final_reason_columns = [
    "alpha_name",
    "horizon",
    "original_promotion_decision",
    "promotion_decision_final",
    "final_status",
    "alpha_role",
    "survivor_selection_score",
    "alpha_behavior_cluster",
    "alpha_sleeve",
    "cluster_rank",
    "cluster_selection_role",
    "cluster_selection_reason",
    "max_corr_to_selected_core",
    "correlated_with_core_alpha",
    "stress_status",
    "source_wfv_status",
]
print("Final decision and rejection/alternate reason for each admitted candidate")
display(
    admitted_final_decisions[[column for column in final_reason_columns if column in admitted_final_decisions.columns]]
    .sort_values(["promotion_decision_final", "survivor_selection_score", "alpha_name"], ascending=[True, False, True])
    .reset_index(drop=True)
)


Final PROMOTE_CORE survivors selected: 1
Cluster summary


,alpha_behavior_cluster,alpha_sleeve,n_candidates,n_promote_core_original,n_review_satellite,n_final_core,best_alpha_name,best_score,best_final_status,avg_abs_corr_with_selected_core,cluster_decision,survivor_version,run_id
0,CORE_REGIME,CORE_REGIME,2,0,2,1,alpha_regime_blend_dynamic_v4_smooth,49.596981,CORE_ALPHA_SURVIVOR,0.990363,FINAL_CORE_SELECTED,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437


Final registry by final_status


,final_status,n_rows
0,CORE_ALPHA_SURVIVOR,1
1,SATELLITE_WATCHLIST,1


Final registry by promotion_decision_final


...

,survivor_id,alpha_name,horizon,alpha_sleeve,original_promotion_decision,promotion_decision_final,final_status,alpha_role,survivor_tier,survivor_selection_score,alpha_behavior_cluster,cluster_rank,cluster_selection_role,cluster_selection_reason,max_corr_to_selected_core,correlated_with_core_alpha,pass_rate,worst_degradation,avg_turnover_proxy,turnover_risk_flag,stress_status,source_wfv_status,failure_category,interpretation_notes,stress_version,alpha_construction_version,date_frozen,survivor_version,run_id,timestamp_frozen
0,phase8_cluster_aware_survivor_v5::alpha_regime...,alpha_regime_blend_dynamic_v4_smooth,20,CORE_REGIME,REVIEW_SATELLITE,PROMOTE_CORE,CORE_ALPHA_SURVIVOR,CORE_ALPHA,WATCH_STRESS_SURVIVOR,49.596981,CORE_REGIME,1,CLUSTER_CORE_SURVIVOR,Best eligible alpha from a distinct behavior c...,<NA>,<NA>,0.888889,0.422319,1.746788,LOW_TURNOVER_RISK,APPROVED_STRESS,WATCHLIST_CONSTRUCTED_ALPHA_WFV,NONE,High average performance but failed catastroph...,phase7_dynamic_alpha_stress_v3,phase4a_alpha_construction_v4,2026-05-11,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437,2026-05-11 08:44:37


Satellite/watchlist/alternate alphas


,survivor_id,alpha_name,horizon,alpha_sleeve,original_promotion_decision,promotion_decision_final,final_status,alpha_role,survivor_tier,survivor_selection_score,alpha_behavior_cluster,cluster_rank,cluster_selection_role,cluster_selection_reason,max_corr_to_selected_core,correlated_with_core_alpha,pass_rate,worst_degradation,avg_turnover_proxy,turnover_risk_flag,stress_status,source_wfv_status,failure_category,interpretation_notes,stress_version,alpha_construction_version,date_frozen,survivor_version,run_id,timestamp_frozen
1,phase8_cluster_aware_survivor_v5::alpha_hybrid...,alpha_hybrid_adaptive_v4_smooth,20,CORE_REGIME,REVIEW_SATELLITE,REVIEW_SATELLITE,SATELLITE_WATCHLIST,SATELLITE_CANDIDATE,WATCH_STRESS_SURVIVOR,37.667225,CORE_REGIME,2,WEAK_SCORE_WATCHLIST,Score below MIN_CLUSTER_SCORE 40.,0.990363,alpha_regime_blend_dynamic_v4_smooth,0.833333,0.678978,1.738334,LOW_TURNOVER_RISK,APPROVED_STRESS,WATCHLIST_CONSTRUCTED_ALPHA_WFV,NONE,High average performance but failed catastroph...,phase7_dynamic_alpha_stress_v3,phase4a_alpha_construction_v4,2026-05-11,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437,2026-05-11 08:44:37


Final decision and rejection/alternate reason for each admitted candidate


,alpha_name,horizon,original_promotion_decision,promotion_decision_final,final_status,alpha_role,survivor_selection_score,alpha_behavior_cluster,alpha_sleeve,cluster_rank,cluster_selection_role,cluster_selection_reason,max_corr_to_selected_core,correlated_with_core_alpha,stress_status,source_wfv_status
0,alpha_regime_blend_dynamic_v4_smooth,20,REVIEW_SATELLITE,PROMOTE_CORE,CORE_ALPHA_SURVIVOR,CORE_ALPHA,49.596981,CORE_REGIME,CORE_REGIME,1,CLUSTER_CORE_SURVIVOR,Best eligible alpha from a distinct behavior c...,<NA>,<NA>,APPROVED_STRESS,WATCHLIST_CONSTRUCTED_ALPHA_WFV
1,alpha_hybrid_adaptive_v4_smooth,20,REVIEW_SATELLITE,REVIEW_SATELLITE,SATELLITE_WATCHLIST,SATELLITE_CANDIDATE,37.667225,CORE_REGIME,CORE_REGIME,2,WEAK_SCORE_WATCHLIST,Score below MIN_CLUSTER_SCORE 40.,0.990363,alpha_regime_blend_dynamic_v4_smooth,APPROVED_STRESS,WATCHLIST_CONSTRUCTED_ALPHA_WFV


## 7. Build reports and pre-ML alpha inputs

In [7]:
survivor_validation_report = validate_correlation_aware_survivor_registry(survivor_registry)
survivor_lineage_report = build_constructed_survivor_lineage_report(
    survivor_registry=survivor_registry,
    construction_metadata=None,
    stress_audit=alpha_stress_audit,
)
survivor_freeze_report = build_survivor_freeze_report(final_core_registry)

pre_ml_alpha_inputs = build_constructed_pre_ml_alpha_inputs(
    alpha_candidates_long=alpha_constructed_candidates,
    survivor_registry=final_core_registry,
)
pre_ml_shape = pd.DataFrame(
    [{"artifact": "pre_ml_alpha_inputs", "rows": len(pre_ml_alpha_inputs), "columns": len(pre_ml_alpha_inputs.columns)}]
)
pre_ml_alpha_names = sorted(pre_ml_alpha_inputs["alpha_name"].dropna().unique().tolist()) if not pre_ml_alpha_inputs.empty else []
pre_ml_core_only_check = set(pre_ml_alpha_names).issubset(set(final_core_registry["alpha_name"].dropna()))

print("Survivor validation report")
display(survivor_validation_report)
print("Survivor freeze report")
display(survivor_freeze_report)
print(f"Pre-ML alpha input rows: {len(pre_ml_alpha_inputs):,}")
print(f"Pre-ML alpha names: {pre_ml_alpha_names}")
print(f"Pre-ML contains only final PROMOTE_CORE alphas: {pre_ml_core_only_check}")
display(pre_ml_shape)
display(pre_ml_alpha_inputs.head())


Survivor validation report


,check_name,passed,details
0,required_columns_exist,True,All required correlation-aware survivor regist...
1,survivor_id_unique,True,Duplicate survivor_id rows: 0
2,at_least_one_final_core,True,Final PROMOTE_CORE rows: 1
3,final_core_originates_from_eligible_stress_dec...,True,Final core rows not originally in ['PROMOTE_CO...
4,no_high_turnover_final_core,True,Final core rows with HIGH_TURNOVER_RISK: 0
5,no_regime_overlay_alpha_names,True,Rows with regime-overlay-like alpha names: 0


Survivor freeze report


,survivor_version,n_survivors,survivor_names,date_frozen,notes
0,phase8_cluster_aware_survivor_v5,1,alpha_regime_blend_dynamic_v4_smooth,2026-05-11,Frozen from stress-approved Notebook 7 alphas;...


Pre-ML alpha input rows: 1,002,844
Pre-ML alpha names: ['alpha_regime_blend_dynamic_v4_smooth']
Pre-ML contains only final PROMOTE_CORE alphas: True


,artifact,rows,columns
0,pre_ml_alpha_inputs,1002844,8


,Date,ticker,alpha_name,alpha_value,horizon,survivor_id,survivor_version,run_id
0,2018-01-02,A,alpha_regime_blend_dynamic_v4_smooth,NaN,20,phase8_cluster_aware_survivor_v5::alpha_regime...,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437
1,2018-01-02,AAPL,alpha_regime_blend_dynamic_v4_smooth,NaN,20,phase8_cluster_aware_survivor_v5::alpha_regime...,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437
2,2018-01-02,ABBV,alpha_regime_blend_dynamic_v4_smooth,NaN,20,phase8_cluster_aware_survivor_v5::alpha_regime...,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437
3,2018-01-02,ABNB,alpha_regime_blend_dynamic_v4_smooth,NaN,20,phase8_cluster_aware_survivor_v5::alpha_regime...,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437
4,2018-01-02,ABT,alpha_regime_blend_dynamic_v4_smooth,NaN,20,phase8_cluster_aware_survivor_v5::alpha_regime...,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437


## 8. Save survivor artifacts to SQLite

In [8]:
saved_paths = save_survivor_outputs(
    survivor_registry=survivor_registry,
    pre_ml_alpha_inputs=pre_ml_alpha_inputs,
    survivor_freeze_report=survivor_freeze_report,
    survivor_validation_report=survivor_validation_report,
    survivor_lineage_report=survivor_lineage_report,
    survivor_alpha_correlation=survivor_alpha_correlation,
    survivor_cluster_summary=survivor_cluster_summary,
    db_path=DB_PATH,
    run_id=run_id,
    survivor_version=SURVIVOR_VERSION,
)

sqlite_tables_written = pd.DataFrame(
    [
        {
            "artifact": artifact,
            "current_table": tables[0],
            "history_table": tables[1],
            "sqlite_path": str(saved_paths[artifact]),
        }
        for artifact, tables in SURVIVOR_TABLES.items()
    ]
)

display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving survivor outputs')


,artifact,current_table,history_table,sqlite_path
0,registry,survivor_alpha_registry_current,survivor_alpha_registry_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,pre_ml_inputs,pre_ml_alpha_inputs_current,pre_ml_alpha_inputs_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,correlation,survivor_alpha_correlation_current,survivor_alpha_correlation_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,cluster_summary,survivor_cluster_summary_current,survivor_cluster_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,freeze_report,survivor_freeze_report_current,survivor_freeze_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,validation_report,survivor_validation_report_current,survivor_validation_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
6,lineage_report,survivor_lineage_report_current,survivor_lineage_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


## 9. Final summary

In [9]:
regime_overlay_name_count = int(
    survivor_registry["alpha_name"].astype(str).str.contains("__|regime_context", case=False, na=False).sum()
)
pre_ml_with_non_core = sorted(
    set(pre_ml_alpha_names).difference(set(final_core_registry["alpha_name"].dropna()))
)

final_summary = pd.DataFrame(
    [
        {"metric": "run_id", "value": run_id},
        {"metric": "timestamp_frozen", "value": timestamp_frozen},
        {"metric": "survivor_version", "value": SURVIVOR_VERSION},
        {"metric": "core_max_abs_correlation", "value": CORE_MAX_ABS_CORRELATION},
        {"metric": "cluster_max_abs_correlation", "value": CLUSTER_MAX_ABS_CORRELATION},
        {"metric": "target_max_core_survivors", "value": TARGET_MAX_CORE_SURVIVORS},
        {"metric": "min_cluster_score", "value": MIN_CLUSTER_SCORE},
        {"metric": "n_primary_candidates", "value": len(primary_candidate_pool)},
        {"metric": "n_pairwise_correlations", "value": len(survivor_alpha_correlation)},
        {"metric": "n_behavior_clusters", "value": primary_candidate_pool["alpha_behavior_cluster"].nunique()},
        {"metric": "n_final_core_survivors", "value": len(final_core_registry)},
        {"metric": "n_registry_rows", "value": len(survivor_registry)},
        {"metric": "pre_ml_alpha_input_rows", "value": len(pre_ml_alpha_inputs)},
        {"metric": "pre_ml_alpha_names", "value": ", ".join(pre_ml_alpha_names)},
        {"metric": "pre_ml_only_final_promote_core", "value": pre_ml_core_only_check and not pre_ml_with_non_core},
        {"metric": "regime_overlay_alpha_name_count", "value": regime_overlay_name_count},
    ]
)

print("Stress candidate counts by promotion_decision")
display(stress_candidate_counts)

print("Behavior cluster counts")
display(behavior_cluster_counts)

print("Cluster summary")
display(survivor_cluster_summary)

print("Survivor selection score table")
display(scored_primary_candidates[[column for column in score_columns if column in scored_primary_candidates.columns]])

print("Correlation matrix")
display(correlation_matrix)

print("Highest correlations")
display(highest_correlations)

print("Final registry by final_status")
display(final_status_counts)

print("Final core selected alphas")
display(final_core_registry)

print("Satellite/watchlist/alternate alphas")
display(retained_alternates)

print("Pre-ML input shape")
display(pre_ml_shape)

print("Confirmation only final PROMOTE_CORE feeds pre_ml")
display(pd.DataFrame([{"pre_ml_only_final_promote_core": pre_ml_core_only_check and not pre_ml_with_non_core, "non_core_names_in_pre_ml": ", ".join(pre_ml_with_non_core)}]))

print("SQLite tables written")
display(sqlite_tables_written)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after saving survivor outputs')

print("Final summary")
display(final_summary)


Stress candidate counts by promotion_decision


,promotion_decision,n_alpha_horizons
0,REVIEW_SATELLITE,2
1,REJECT,1


Behavior cluster counts


,alpha_behavior_cluster,n_alpha_horizons
0,CORE_REGIME,2


Cluster summary


...

,pre_ml_only_final_promote_core,non_core_names_in_pre_ml
0,True,


SQLite tables written


,artifact,current_table,history_table,sqlite_path
0,registry,survivor_alpha_registry_current,survivor_alpha_registry_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
1,pre_ml_inputs,pre_ml_alpha_inputs_current,pre_ml_alpha_inputs_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
2,correlation,survivor_alpha_correlation_current,survivor_alpha_correlation_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
3,cluster_summary,survivor_cluster_summary_current,survivor_cluster_summary_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
4,freeze_report,survivor_freeze_report_current,survivor_freeze_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
5,validation_report,survivor_validation_report_current,survivor_validation_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...
6,lineage_report,survivor_lineage_report_current,survivor_lineage_report_history,/Users/AnyiXu_1/Desktop/multi-factor-equity-al...


Final summary


,metric,value
0,run_id,phase8_cluster_aware_survivor_20260511_084437
1,timestamp_frozen,2026-05-11 08:44:37
2,survivor_version,phase8_cluster_aware_survivor_v5
3,core_max_abs_correlation,0.9
4,cluster_max_abs_correlation,0.95
5,target_max_core_survivors,4
6,min_cluster_score,40
7,n_primary_candidates,2
8,n_pairwise_correlations,1
9,n_behavior_clusters,1


In [10]:
from src.db import load_table

registry = load_table("survivor_alpha_registry_current")
corr = load_table("survivor_alpha_correlation_current")
cluster_summary = load_table("survivor_cluster_summary_current")
pre_ml = load_table("pre_ml_alpha_inputs_current")

registry_display_columns = [
    "alpha_name",
    "alpha_sleeve",
    "original_promotion_decision",
    "promotion_decision_final",
    "final_status",
    "alpha_role",
    "survivor_selection_score",
    "alpha_behavior_cluster",
    "cluster_rank",
    "cluster_selection_role",
    "cluster_selection_reason",
    "max_corr_to_selected_core",
    "correlated_with_core_alpha",
]
display(registry[[column for column in registry_display_columns if column in registry.columns]])

display(cluster_summary)
display(corr.sort_values("abs_correlation", ascending=False).head(20))

display(pre_ml["alpha_name"].value_counts())
print("pre_ml shape:", pre_ml.shape)

gc.collect()
if 'cleanup_memory' in globals():
    cleanup_memory('after survivor diagnostics readback')


,alpha_name,alpha_sleeve,original_promotion_decision,promotion_decision_final,final_status,alpha_role,survivor_selection_score,alpha_behavior_cluster,cluster_rank,cluster_selection_role,cluster_selection_reason,max_corr_to_selected_core,correlated_with_core_alpha
0,alpha_regime_blend_dynamic_v4_smooth,CORE_REGIME,REVIEW_SATELLITE,PROMOTE_CORE,CORE_ALPHA_SURVIVOR,CORE_ALPHA,49.596981,CORE_REGIME,1,CLUSTER_CORE_SURVIVOR,Best eligible alpha from a distinct behavior c...,None,None
1,alpha_hybrid_adaptive_v4_smooth,CORE_REGIME,REVIEW_SATELLITE,REVIEW_SATELLITE,SATELLITE_WATCHLIST,SATELLITE_CANDIDATE,37.667225,CORE_REGIME,2,WEAK_SCORE_WATCHLIST,Score below MIN_CLUSTER_SCORE 40.,0.990362896603554,alpha_regime_blend_dynamic_v4_smooth


,alpha_behavior_cluster,alpha_sleeve,n_candidates,n_promote_core_original,n_review_satellite,n_final_core,best_alpha_name,best_score,best_final_status,avg_abs_corr_with_selected_core,cluster_decision,survivor_version,run_id
0,CORE_REGIME,CORE_REGIME,2,0,2,1,alpha_regime_blend_dynamic_v4_smooth,49.596981,CORE_ALPHA_SURVIVOR,0.990363,FINAL_CORE_SELECTED,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437


,alpha_name_1,alpha_name_2,correlation,abs_correlation,alpha_1_promotion_decision,alpha_2_promotion_decision,alpha_1_pass_rate,alpha_2_pass_rate,alpha_1_worst_degradation,alpha_2_worst_degradation,survivor_version,run_id
0,alpha_hybrid_adaptive_v4_smooth,alpha_regime_blend_dynamic_v4_smooth,0.990363,0.990363,REVIEW_SATELLITE,REVIEW_SATELLITE,0.833333,0.888889,0.678978,0.422319,phase8_cluster_aware_survivor_v5,phase8_cluster_aware_survivor_20260511_084437


alpha_name
alpha_regime_blend_dynamic_v4_smooth    1002844
Name: count, dtype: int64

pre_ml shape: (1002844, 8)
